# 12 — Baseline Validation + Interpretable Quality Stories

Two questions:
1. **Does team context actually help?** Compare pre-player-only baseline vs full model with team features.
2. **What specific stories emerge?** Individual regressions per quality — find the strongest, most interpretable predictions.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'
raw = pd.read_parquet(f'{V1}/model_df.parquet')

In [ ]:
df = raw[raw['position'] != 'Goalkeeper'].copy()
team_cols = [c for c in df.columns if c.startswith('from_q_proj_') or c.startswith('to_q_') or c.startswith('delta_team_')]
df = df[~df[team_cols].isna().any(axis=1)].copy()

ALL_PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]
TEAM_ALL = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
            'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']
POSITIONS = sorted(df['position'].unique())

position_qualities = {}
for pos in POSITIONS:
    mask = df['position'] == pos
    position_qualities[pos] = [q for q in ALL_PLAYER_QUALITIES if df.loc[mask, f'pre_{q}'].isna().mean() < 0.50]

# Style distance
from_cols = [f'from_q_proj_{q}' for q in TEAM_ALL]
to_cols = [f'to_q_{q}' for q in TEAM_ALL]
df['style_distance'] = np.sqrt(((df[to_cols].values - df[from_cols].values) ** 2).sum(axis=1))

train_mask = df['to_season'].isin([2020, 2021, 2022, 2023])
test_mask = df['to_season'] == 2024
print(f'{len(df):,} transfers | Train: {train_mask.sum():,} | Test: {test_mask.sum():,}')

## 1. Baseline: Does Team Context Matter?

**Baseline**: only pre-player qualities predict post-player qualities.  
**Full model**: + origin team style, destination team style, style distance, pre-minutes.

In [ ]:
ALPHAS = np.logspace(-3, 1, 20)

baseline_results = []

for pos in POSITIONS:
    qualities = position_qualities[pos]
    pre_cols = [f'pre_{q}' for q in qualities]
    team_features = ([f'from_q_proj_{q}' for q in TEAM_ALL] +
                     [f'to_q_{q}' for q in TEAM_ALL] +
                     ['style_distance', 'pre_minutes'])
    full_cols = pre_cols + team_features

    pos_df = df[df['position'] == pos]

    for q in qualities:
        target = f'post_{q}'
        # Clean data (need full_cols + target non-null)
        needed = full_cols + [target]
        clean = pos_df[needed].dropna()
        train_idx = clean.index[clean.index.isin(df[train_mask].index)]
        test_idx = clean.index[clean.index.isin(df[test_mask].index)]

        if len(train_idx) < 20 or len(test_idx) < 5:
            continue

        y_train = clean.loc[train_idx, target].values
        y_test = clean.loc[test_idx, target].values

        # Baseline: pre only
        X_tr_base = clean.loc[train_idx, pre_cols].values
        X_te_base = clean.loc[test_idx, pre_cols].values
        m_base = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X_tr_base, y_train)
        r2_base = r2_score(y_test, m_base.predict(X_te_base))

        # Full: pre + team + FE
        X_tr_full = clean.loc[train_idx, full_cols].values
        X_te_full = clean.loc[test_idx, full_cols].values
        m_full = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X_tr_full, y_train)
        r2_full = r2_score(y_test, m_full.predict(X_te_full))

        baseline_results.append({
            'position': pos, 'quality': q,
            'r2_baseline': r2_base, 'r2_full': r2_full,
            'delta_r2': r2_full - r2_base,
            'n_train': len(train_idx), 'n_test': len(test_idx),
        })

    print(f'  {pos}: done')

bl_df = pd.DataFrame(baseline_results)
print(f'\n{len(bl_df)} quality-position combos evaluated')

In [ ]:
# Summary: mean R² baseline vs full per position
pos_comp = bl_df.groupby('position')[['r2_baseline', 'r2_full', 'delta_r2']].mean().round(4)
print('MEAN R\u00b2: BASELINE vs FULL')
print(pos_comp.to_string())
print(f'\nOverall: baseline={bl_df["r2_baseline"].mean():.4f}, full={bl_df["r2_full"].mean():.4f}, \u0394={bl_df["delta_r2"].mean():.4f}')

In [ ]:
# Heatmap: delta R² per quality x position
pivot_delta = bl_df.pivot_table(index='quality', columns='position', values='delta_r2')

fig = px.imshow(
    pivot_delta.round(3),
    text_auto='.3f',
    color_continuous_scale='RdBu',
    zmin=-0.1, zmax=0.1,
    labels=dict(x='Position', y='Quality', color='\u0394 R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title='\u0394 R\u00b2 (Full \u2212 Baseline) \u2014 Where Team Context Helps',
    height=600, width=650,
    margin=dict(t=50, b=30, l=170, r=30),
)
fig.show()

In [ ]:
# Grouped bar: baseline vs full for top 10 biggest delta combos
top_delta = bl_df.nlargest(12, 'delta_r2').copy()
top_delta['label'] = top_delta['position'].str[:3] + ' — ' + top_delta['quality']
top_delta = top_delta.sort_values('delta_r2', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=top_delta['label'], x=top_delta['r2_baseline'], name='Baseline (pre only)',
                      orientation='h', marker_color='#636EFA'))
fig.add_trace(go.Bar(y=top_delta['label'], x=top_delta['r2_full'], name='Full (+ team context)',
                      orientation='h', marker_color='#00CC96'))
fig.update_layout(
    title='Top 12 Qualities Where Team Context Adds Most Value',
    barmode='group', height=450, width=750,
    margin=dict(t=50, b=30, l=160, r=30),
    xaxis_title='R\u00b2',
    legend=dict(x=0.6, y=0.05),
)
fig.show()

## 2. Individual Quality Regressions

One Lasso per target quality (not multi-output). Each quality gets its own optimal regularization and feature selection.

In [ ]:
# Fit individual Lasso per (position, quality) — store models + detailed results
individual_results = []
fitted_models = {}

for pos in POSITIONS:
    qualities = position_qualities[pos]
    pre_cols = [f'pre_{q}' for q in qualities]
    team_features = ([f'from_q_proj_{q}' for q in TEAM_ALL] +
                     [f'to_q_{q}' for q in TEAM_ALL] +
                     ['style_distance', 'pre_minutes'])
    x_cols = pre_cols + team_features
    pos_df = df[df['position'] == pos]

    for q in qualities:
        target = f'post_{q}'
        clean = pos_df[x_cols + [target]].dropna()
        train_idx = clean.index[clean.index.isin(df[train_mask].index)]
        test_idx = clean.index[clean.index.isin(df[test_mask].index)]

        if len(train_idx) < 20 or len(test_idx) < 5:
            continue

        X_train = clean.loc[train_idx, x_cols].values
        X_test = clean.loc[test_idx, x_cols].values
        y_train = clean.loc[train_idx, target].values
        y_test = clean.loc[test_idx, target].values

        model = LassoCV(alphas=ALPHAS, max_iter=5000).fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)

        # Feature analysis
        coefs = pd.Series(model.coef_, index=x_cols)
        nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)
        team_nonzero = nonzero[[c for c in nonzero.index if c not in pre_cols]]

        top5 = nonzero.head(5)
        top5_str = ', '.join([f'{n.replace("from_q_proj_","from ").replace("to_q_","to ").replace("pre_","pre ")}({v:+.3f})'
                              for n, v in top5.items()])

        individual_results.append({
            'position': pos, 'quality': q, 'r2': r2, 'mae': mae,
            'n_features_kept': (coefs != 0).sum(),
            'n_team_features_kept': len(team_nonzero),
            'has_team_signal': len(team_nonzero) > 0,
            'top5_features': top5_str,
            'alpha': model.alpha_,
            'n_train': len(train_idx), 'n_test': len(test_idx),
        })

        fitted_models[(pos, q)] = {
            'model': model, 'x_cols': x_cols, 'y_test': y_test,
            'y_pred': y_pred, 'coefs': coefs,
        }

ind_df = pd.DataFrame(individual_results).sort_values('r2', ascending=False)
print(f'{len(ind_df)} individual models fitted')
print(f'\nR\u00b2 distribution: mean={ind_df["r2"].mean():.3f}, median={ind_df["r2"].median():.3f}')
print(f'Models with R\u00b2 > 0.50: {(ind_df["r2"] > 0.50).sum()}')
print(f'Models with R\u00b2 > 0.60: {(ind_df["r2"] > 0.60).sum()}')

In [ ]:
# Heatmap: R² per quality x position (individual models)
pivot_ind = ind_df.pivot_table(index='quality', columns='position', values='r2')

fig = px.imshow(
    pivot_ind.round(3),
    text_auto='.3f',
    color_continuous_scale='RdYlGn',
    zmin=-0.1, zmax=0.8,
    labels=dict(x='Position', y='Quality', color='R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title='R\u00b2 per Quality x Position (individual Lasso models)',
    height=600, width=650,
    margin=dict(t=50, b=30, l=170, r=30),
)
fig.show()

## 3. The Stories — Top Findings

The strongest, most interpretable predictions. Each one answers: *"What happens to quality X when a player moves to a team with style Y?"*

In [ ]:
# Top models with R² > 0.50
stories = ind_df[ind_df['r2'] > 0.50].copy()
print(f'{len(stories)} models with R\u00b2 > 0.50\n')
print(stories[['position', 'quality', 'r2', 'mae', 'n_features_kept', 'top5_features']].to_string(index=False))

In [ ]:
# Scatter plots: predicted vs actual for top 8 models
top_stories = stories.head(8)
n_plots = len(top_stories)
cols = 2
rows = (n_plots + 1) // 2

fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{r['position'][:3]} — {r['quality']} (R\u00b2={r['r2']:.3f})" for _, r in top_stories.iterrows()],
    horizontal_spacing=0.08, vertical_spacing=0.08,
)

for idx, (_, row) in enumerate(top_stories.iterrows()):
    r_idx = idx // cols + 1
    c_idx = idx % cols + 1
    key = (row['position'], row['quality'])
    info = fitted_models[key]

    fig.add_trace(go.Scatter(
        x=info['y_test'], y=info['y_pred'],
        mode='markers', marker=dict(size=4, opacity=0.5, color='#636EFA'),
        showlegend=False,
    ), row=r_idx, col=c_idx)

    # Perfect prediction line
    mn = min(info['y_test'].min(), info['y_pred'].min())
    mx = max(info['y_test'].max(), info['y_pred'].max())
    fig.add_trace(go.Scatter(
        x=[mn, mx], y=[mn, mx],
        mode='lines', line=dict(dash='dash', color='gray', width=1),
        showlegend=False,
    ), row=r_idx, col=c_idx)

fig.update_layout(
    title='Predicted vs Actual — Top Individual Models',
    height=250 * rows, width=750,
    margin=dict(t=60, b=30, l=50, r=30),
)
fig.show()

In [ ]:
# Deep dive: coefficient breakdown for top stories
for _, row in top_stories.iterrows():
    key = (row['position'], row['quality'])
    info = fitted_models[key]
    coefs = info['coefs']
    nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)

    # Shorten names
    short = nonzero.copy()
    short.index = (short.index
        .str.replace('from_q_proj_', 'from ', regex=False)
        .str.replace('to_q_', 'to ', regex=False)
        .str.replace('pre_', 'pre ', regex=False))

    top10 = short.head(10)
    colors = ['#EF553B' if v < 0 else '#00CC96' for v in top10.values]

    fig = go.Figure(go.Bar(
        x=top10.values, y=top10.index,
        orientation='h', marker_color=colors,
    ))
    fig.update_layout(
        title=f"{row['position']} — post {row['quality']} (R\u00b2={row['r2']:.3f})",
        height=max(300, len(top10) * 30),
        width=600,
        margin=dict(t=40, b=30, l=200, r=30),
        xaxis_title='Coefficient',
        yaxis=dict(autorange='reversed'),
    )
    fig.show()

## 4. Summary

In [ ]:
# Full summary table
summary = bl_df.merge(ind_df[['position', 'quality', 'r2', 'n_features_kept', 'top5_features']],
                       on=['position', 'quality'], suffixes=('', '_ind'))
summary = summary.rename(columns={'r2': 'r2_individual'})
summary = summary.sort_values('r2_individual', ascending=False)

print('FULL RESULTS (sorted by individual R\u00b2)')
print(summary[['position', 'quality', 'r2_baseline', 'r2_full', 'delta_r2',
               'r2_individual', 'n_features_kept']].head(20).to_string(index=False))

In [ ]:
# Final conclusions
mean_base = bl_df['r2_baseline'].mean()
mean_full = bl_df['r2_full'].mean()
mean_ind = ind_df['r2'].mean()
pct_positive_delta = (bl_df['delta_r2'] > 0).mean() * 100
n_strong = (ind_df['r2'] > 0.50).sum()

print('CONCLUSIONS')
print('=' * 70)
print(f'\n1. DOES TEAM CONTEXT MATTER?')
print(f'   Baseline (pre only):     mean R\u00b2 = {mean_base:.4f}')
print(f'   Full (+ team context):   mean R\u00b2 = {mean_full:.4f}')
print(f'   \u0394 R\u00b2:                    {mean_full - mean_base:+.4f}')
print(f'   Team context improved {pct_positive_delta:.0f}% of quality-position combos')
print(f'\n2. INDIVIDUAL MODELS')
print(f'   Mean R\u00b2 (individual Lasso): {mean_ind:.4f}')
print(f'   Models with R\u00b2 > 0.50: {n_strong} / {len(ind_df)}')
print(f'   Best: {ind_df.iloc[0]["position"]} — {ind_df.iloc[0]["quality"]} (R\u00b2={ind_df.iloc[0]["r2"]:.3f})')